# NHA Hackathon – Problem Statement 01  
## Clinical Document Classification & Compliance to STG requirements

This notebook is a **starter scaffold** for participants working on:

- mixed-quality healthcare document ingestion
- OCR + layout understanding
- visual cue detection
- STG / policy rule checks
- explainable claim decisioning
- episode timeline construction
- extra / non-required document identification


### Deliverables this notebook is designed to help produce
1. **Per-page/package JSON output** in the exact format required by the problem statement  
2. **Human-readable summary table** with document type, rule checks, and reasons  
3. **Episode timeline** with admission / investigation / procedure / discharge ordering  
4. **Decision**: `PASS`, `CONDITIONAL`, or `FAIL` with evidence and confidence

This notebook is a skeleton and can be changed by participants 

In [1]:
# =========================
# 1. INSTALLS / IMPORTS
# =========================

# Uncomment and adapt as needed during hackathon usage.
# !pip install pymupdf pdf2image pillow opencv-python pandas numpy pydantic python-dateutil rapidfuzz

from __future__ import annotations

import os
import re
import json
import math
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import pandas as pd
import numpy as np

## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS1: c110a5f8-6e79-43bd-bd7a-979677354958**

In [2]:
from databank_download_widget import DatabankDownloadWidget

# Create an instance of the databank widget
databank_downloader = DatabankDownloadWidget()

# Display the widget
databank_downloader.display()

ModuleNotFoundError: No module named 'databank_download_widget'

In [ ]:
# =========================
# 2. CONFIG
# =========================

DATA_ROOT = Path("./Claims")         # input claims folder
OUTPUT_ROOT = Path("./outputs")    # json/csv/html outputs
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

PACKAGE_CODES = ["MG064A", "SG039C", "MG006A", "SB039A"]

DECISION_PASS = "PASS"
DECISION_CONDITIONAL = "CONDITIONAL"
DECISION_FAIL = "FAIL"

### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere

In [ ]:
from nha_client import NHAclient
import base64


clientId=""
clientSecret=""


nc = NHAclient(clientId, clientSecret)


with open("Sample.jpg", "rb") as image_file:
    image_bytes = image_file.read()

image_base64 = base64.b64encode(image_bytes).decode("utf-8")
data_url = f"data:image/jpeg;base64,{image_base64}"

response = nc.completion(
    model="", #use one of the models
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text", "text": "What do you see"},
            ],
        }
    ],
    metadata={
            "problem_statement":1
        }
)

print(response)

In [ ]:
# =========================
# 3. OUTPUT SCHEMAS
# =========================
# These are the exact expected keys per package, based on the provided examples.

PACKAGE_SCHEMAS = {
    "MG064A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "cbc_hb_report", "indoor_case",
        "treatment_details", "post_hb_report", "discharge_summary",
        "severe_anemia", "common_signs", "significant_signs",
        "life_threatening_signs", "extra_document", "document_rank"
    ],
    "SG039C": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "usg_report", "lft_report", "operative_notes",
        "pre_anesthesia", "discharge_summary", "photo_evidence",
        "histopathology", "clinical_condition", "usg_calculi",
        "pain_present", "previous_surgery", "extra_document", "document_rank"
    ],
    "MG006A": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "investigation_pre", "pre_date", "vitals_treatment",
        "investigation_post", "post_date", "discharge_summary", "poor_quality",
        "fever", "symptoms", "extra_document", "document_rank"
    ],
    "SB039A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes",
        "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary",
        "doa", "dod", "arthritis_type", "post_op_implant_present",
        "age_valid", "extra_document", "document_rank"
    ],
}

KEY_ALIASES = {
    "S3_link": "link",
    "s3_link": "link",
    "S3_link/DocumentName": "link",
}

In [ ]:
# =========================
# 4. DATA MODELS
# =========================

@dataclass
class OCRLine:
    text: str
    bbox: Optional[List[int]] = None
    confidence: Optional[float] = None

@dataclass
class PageResult:
    case_id: str
    file_name: str
    page_number: int
    extracted_text: str = ""
    ocr_lines: List[OCRLine] = field(default_factory=list)
    doc_type: str = "unknown"
    doc_type_confidence: float = 0.0
    visual_tags: Dict[str, Any] = field(default_factory=dict)
    entities: Dict[str, Any] = field(default_factory=dict)
    quality: Dict[str, Any] = field(default_factory=dict)
    output_row: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TimelineEvent:
    sequence: int
    event_type: str
    date: Optional[str]
    source_document: str
    temporal_validity: str
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ClaimDecision:
    case_id: str
    package_code: str
    decision: str
    confidence: float
    reasons: List[str]
    missing_documents: List[str] = field(default_factory=list)
    rule_flags: List[str] = field(default_factory=list)
    timeline_flags: List[str] = field(default_factory=list)

## Recommended pipeline stages

1. **Ingest** claim files  
2. **Split** PDFs/images into pages  
3. **OCR** each page  
4. **Layout / document type classification**  
5. **Visual cue detection** (stamp/signature/QR/barcode/implant sticker/photo evidence)  
6. **Entity extraction** (dates, diagnosis, procedure, age, amounts, etc.)  
7. **Package-specific row creation**  
8. **Timeline construction**  
9. **Rules engine**  
10. **Explainable decisioning**

In [ ]:
# =========================
# 1. INGEST CLAIM FILES
# =========================

def iter_case_files(case_dir: Path) -> List[Path]:
    '''
    Discover all supported files inside a single case directory.

    Expected behavior when implemented:
    - recursively scan the case folder
    - keep only files with supported claim-document extensions
    - return a deterministic, sorted list of paths for downstream processing
    - preserve the raw file paths exactly as submitted by the participant input
    '''
    pass

def discover_cases(data_root: Path) -> Dict[str, List[Path]]:
    '''
    Build the input mapping from case identifier to its associated files.

    Expected behavior when implemented:
    - scan the root input directory
    - treat each child case folder as one claim/case
    - collect supported files for each case using iter_case_files(...)
    - return a dictionary of the form:
      {case_id: [file_path_1, file_path_2, ...]}
    '''
    pass


In [ ]:
# =========================
# 2. SPLIT PDFS/IMAGES INTO PAGES
# =========================

def extract_pages(file_path: Path) -> List[Dict[str, Any]]:
    '''
    Convert an input file into page-level image records for the pipeline.

    Expected behavior when implemented:
    - if the file is a PDF, rasterize each page into an image
    - if the file is already an image, wrap it as a single page
    - preserve the original filename and assign sequential page numbers
    - return a list of dictionaries such as:
      [{"page_number": 1, "image": <page_image>, "file_name": "..."}]
    '''
    pass


In [ ]:
# =========================
# 3. OCR EACH PAGE
# =========================

def run_ocr(page_image: Any) -> Tuple[str, List[OCRLine]]:
    '''
    Extract text and line-level OCR evidence from a single page image.

    Expected behavior when implemented:
    - run OCR on the provided page image
    - return the full extracted text as a single string
    - return structured OCR lines with optional bounding boxes/confidence
    - support mixed-quality and multilingual claim documents where possible
    '''
    pass


In [ ]:
# =========================
# 4. LAYOUT / DOCUMENT TYPE CLASSIFIER
# =========================

def estimate_page_quality(page_image: Any, extracted_text: str) -> Dict[str, Any]:
    '''
    Estimate whether a page is usable and quantify key quality issues.

    Expected behavior when implemented:
    - assess blur, skew, noise, cropping, low contrast, or low text density
    - distinguish between poor-quality but usable pages and unreadable pages
    - produce a dictionary of quality indicators consumed by later stages
    - keep output keys aligned with the rest of the notebook
    '''
    pass


In [ ]:
# =========================
# 5. VISUAL CUE DETECTION
# =========================

def detect_visual_elements(page_image: Any) -> Dict[str, Any]:
    '''
    Detect non-text visual evidence required for claim validation.

    Expected behavior when implemented:
    - identify cues such as hospital stamp, signature, QR/barcode,
      implant sticker, or package-specific photo evidence
    - return a dictionary of normalized binary/structured visual tags
    - preserve compatibility with the downstream row-mapping logic
    '''
    pass


In [ ]:
# =========================
# 6. ENTITY EXTRACTION
# =========================

DOCUMENT_TYPES = [
    "clinical_notes",
    "cbc_hb_report",
    "indoor_case",
    "treatment_details",
    "post_hb_report",
    "discharge_summary",
    "usg_report",
    "lft_report",
    "operative_notes",
    "pre_anesthesia",
    "histopathology",
    "xray_ct_knee",
    "investigation_pre",
    "investigation_post",
    "vitals_treatment",
    "implant_invoice",
    "post_op_photo",
    "post_op_xray",
    "photo_evidence",
    "extra_document",
]

def classify_document_type(extracted_text: str, visual_tags: Dict[str, Any]) -> Tuple[str, float]:
    '''
    Assign a document-type label to the page using text, layout, and/or vision cues.

    Expected behavior when implemented:
    - classify a page into one of the normalized document labels in DOCUMENT_TYPES
    - optionally combine OCR text, visual tags, layout cues, and VLM prompting
    - return both the predicted label and a confidence score
    - keep unknown/ambiguous pages explicitly identifiable
    '''
    pass


In [ ]:
# =========================
# ENTITY EXTRACTION HELPERS
# =========================

DATE_PATTERNS = [
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
    r"\b\d{1,2}-[A-Za-z]{3}-\d{2,4}\b",
    r"\b\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4}\b",
]

def find_dates(text: str) -> List[str]:
    '''
    Extract candidate date strings from OCR text.

    Expected behavior when implemented:
    - scan the text using one or more date patterns
    - return unique dates in the order they are found
    - keep raw extracted values so they can be normalized later
    '''
    pass

def find_age(text: str) -> Optional[int]:
    '''
    Extract a candidate age value from free text.

    Expected behavior when implemented:
    - search OCR text for age mentions such as years/yrs
    - return the parsed integer age when found
    - return None when no reliable age mention is available
    '''
    pass

def contains_any(text: str, keywords: List[str]) -> int:
    '''
    Check whether any package-specific keyword appears in the text.

    Expected behavior when implemented:
    - normalize the text and keyword list for robust matching
    - return 1 when any keyword is present, otherwise 0
    - serve as a lightweight helper for package-specific row population
    '''
    pass


In [ ]:
# =========================
# 7. PAGE-TO-ROW MAPPING
# =========================

def populate_row_for_package(
    package_code: str,
    page_result: PageResult,
) -> Dict[str, Any]:
    """
    Create and populate a single output row for one page, based on the assigned package code and the intermediate page-level analysis result.

    Intended responsibilities:
    - Initialize an output row using the case ID, file name, page number, and package code.
    - Map detected document type to the corresponding package-specific presence field when applicable.
    - Populate package-specific clinical, procedural, temporal, and visual fields using extracted text, detected visual tags, and quality signals.
    - Apply package-specific heuristics or rules for values such as clinical condition flags, symptom flags, dates, implant evidence, age validation, and other STG-relevant attributes.
    - Mark whether the page belongs to an extra/non-required document.
    - Assign a document rank based on page role in the episode timeline, or assign rank 99 for extra documents.
    - Return the final row as a dictionary matching the required output schema.

    Notes for participants:
    - Keep the final keys and output structure exactly aligned with the expected evaluation format.
    - Replace starter heuristics with robust logic driven by OCR, document classification, image understanding, and STG-aware rules.
    - Ensure date extraction and package-specific logic remain explainable and reproducible.
    """
    pass

In [ ]:
# =========================
# PACKAGE ROW INITIALIZERS
# =========================

def normalize_output_key(key: str) -> str:
    '''
    Resolve any key aliases so internal logic and required output schema stay aligned.

    Expected behavior when implemented:
    - map alternate/internal key names to the exact organizer-facing keys
    - preserve the strict output format required for evaluation
    '''
    pass

def initialize_output_row(package_code: str, case_id: str, file_name: str, page_number: int) -> Dict[str, Any]:
    '''
    Create an empty page-level output row for a given package code.

    Expected behavior when implemented:
    - initialize all required keys for the selected package schema
    - preserve exact key names and key order
    - fill package metadata such as case id, file name, procedure code, and page number
    - initialize nullable date fields correctly where applicable
    '''
    pass


In [ ]:
# =========================
# PAGE-TO-ROW MAPPING
# =========================

def populate_row_for_package(
    package_code: str,
    page_result: PageResult,
) -> Dict[str, Any]:
    '''
    Convert a processed page into one exact output row for the assigned package.

    Expected behavior when implemented:
    - map document-type predictions into presence/absence fields
    - populate package-specific clinical, visual, and temporal fields
    - identify extra documents where applicable
    - assign the appropriate document rank while keeping the exact output format
    '''
    pass


In [ ]:
# =========================
# DOCUMENT RANKING
# =========================

RANK_MAP = {
    "MG064A": {
        "clinical_notes": 1,
        "cbc_hb_report": 2,
        "indoor_case": 2,
        "treatment_details": 3,
        "post_hb_report": 4,
        "discharge_summary": 5,
    },
    "SG039C": {
        "clinical_notes": 1,
        "usg_report": 2,
        "lft_report": 3,
        "pre_anesthesia": 4,
        "operative_notes": 5,
        "discharge_summary": 5,
        "histopathology": 6,
        "photo_evidence": 6,
    },
    "MG006A": {
        "clinical_notes": 1,
        "investigation_pre": 2,
        "vitals_treatment": 3,
        "investigation_post": 4,
        "discharge_summary": 5,
    },
    "SB039A": {
        "clinical_notes": 1,
        "xray_ct_knee": 2,
        "indoor_case": 3,
        "operative_notes": 4,
        "implant_invoice": 5,
        "post_op_photo": 6,
        "post_op_xray": 6,
        "discharge_summary": 7,
    }
}

def infer_document_rank(package_code: str, row: Dict[str, Any], doc_type: str) -> Optional[int]:
    '''
    Determine the page/document position in the expected clinical timeline.

    Expected behavior when implemented:
    - assign the package-specific rank based on document type and context
    - return rank 99 for extra documents
    - keep the same rank across all pages belonging to the same logical document where required
    '''
    pass


In [ ]:
# =========================
# 8. TIMELINE CONSTRUCTION
# =========================

def build_episode_timeline(package_code: str, page_results: List[PageResult]) -> List[TimelineEvent]:
    '''
    Construct a chronological episode timeline from page-level evidence.

    Expected behavior when implemented:
    - extract admission, investigation, procedure, monitoring, and discharge events
    - order them chronologically or by inferred clinical sequence
    - capture source-document provenance for each event
    - mark basic temporal validity such as before procedure / after treatment / valid
    '''
    pass


In [ ]:
# =========================
# 9. RULES ENGINE (STARTER)
# =========================

MANDATORY_DOCS = {
    "MG064A": ["clinical_notes", "cbc_hb_report", "indoor_case", "treatment_details", "post_hb_report", "discharge_summary"],
    "SG039C": ["clinical_notes", "usg_report", "lft_report", "operative_notes", "pre_anesthesia", "discharge_summary", "photo_evidence", "histopathology"],
    "MG006A": ["clinical_notes", "investigation_pre", "vitals_treatment", "investigation_post", "discharge_summary"],
    "SB039A": ["clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes", "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary"],
}

def aggregate_case_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    '''
    Collapse page-level output rows into case-level evidence signals.

    Expected behavior when implemented:
    - combine per-page binary flags into case-level presence/absence indicators
    - preserve useful date/metadata fields for downstream rules evaluation
    - make the result suitable for package-level adjudication logic
    '''
    pass

def run_rules_engine(case_id: str, package_code: str, rows: List[Dict[str, Any]], timeline: List[TimelineEvent]) -> ClaimDecision:
    '''
    Evaluate the case against package-specific STG and policy rules.

    Expected behavior when implemented:
    - check mandatory-document completeness
    - check package-specific clinical and temporal plausibility
    - generate Pass / Conditional / Fail with reasons and confidence
    - preserve explainability through explicit flags and missing-document lists
    '''
    pass


In [ ]:
# =========================
# 10. EXPLAINABLE DECISIONING
# =========================

def build_human_readable_summary(package_code: str, page_results: List[PageResult], decision: ClaimDecision) -> pd.DataFrame:
    '''
    Build the reviewer-facing summary table for the case.

    Expected behavior when implemented:
    - produce a page/document-level summary aligned with the sample output style
    - show document classification, evidence usage, and rule-related notes
    - remain easy for PPD/CPD reviewers to read
    '''
    pass

def build_timeline_df(timeline: List[TimelineEvent]) -> pd.DataFrame:
    '''
    Convert structured timeline events into a tabular reviewer-friendly format.

    Expected behavior when implemented:
    - preserve sequence, event type, date, source document, and temporal validity
    - produce a DataFrame aligned with the expected timeline output structure
    '''
    pass


In [ ]:
# =========================
# CORE PIPELINE DRIVER
# =========================

def process_case(case_id: str, files: List[Path], package_code: str) -> Dict[str, Any]:
    '''
    Run the full page-level and case-level pipeline for a single claim case.

    Expected behavior when implemented:
    - ingest files and split them into pages
    - run OCR, quality checks, visual detection, and document classification
    - convert page evidence into exact output rows
    - build the episode timeline, run rules, and prepare reviewer-facing outputs
    - return a dictionary containing both strict evaluation outputs and readable summaries
    '''
    pass


In [ ]:
# =========================
# BATCH RUNNER
# =========================

def run_batch(data_root: Path, package_code_lookup: Optional[Dict[str, str]] = None) -> Dict[str, Any]:
    '''
    Execute the claim-validation pipeline across multiple cases.

    Expected behavior when implemented:
    - discover available cases from the input root
    - resolve the package code for each case
    - call process_case(...) for each valid case
    - collect all outputs into a single batch result dictionary
    '''
    pass


In [ ]:
# =========================
# DEMO WITH THE PROVIDED EXAMPLE JSON STRUCTURES
# =========================

EXAMPLE_JSON_PATHS = {
    "SG039C": "data/SG039C_Cholecystectomy.json",
    "SB039A": "data/SB039A_Knee_Replacement.json",
    "MG064A": "data/MG064A_Anemia.json",
    "MG006A": "data/MG006A_Fever.json",
}

def load_example_jsons() -> Dict[str, List[Dict[str, Any]]]:
    out = {}
    for pkg, path in EXAMPLE_JSON_PATHS.items():
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                out[pkg] = json.load(f)
    return out

example_jsons = load_example_jsons()
{k: len(v) for k, v in example_jsons.items()}

In [ ]:
# =========================
# EXPORTERS
# =========================

def export_case_outputs(case_result: Dict[str, Any], output_root: Path = OUTPUT_ROOT) -> None:
    '''
    Write strict outputs and reviewer-facing artifacts to disk.

    Expected behavior when implemented:
    - export the exact package-specific JSON rows for evaluation
    - export the human-readable summary and timeline tables
    - export the explainable decision record for the case
    - preserve the existing output folder structure used by this notebook
    '''
    pass


In [ ]:
# =========================
# VALIDATOR FOR EXACT OUTPUT KEYS
# =========================

def validate_output_rows(package_code: str, rows: List[Dict[str, Any]]) -> Tuple[bool, List[str]]:
    expected = PACKAGE_SCHEMAS[package_code]
    issues = []

    for i, row in enumerate(rows):
        row_keys = list(row.keys())
        if row_keys != expected:
            issues.append(
                f"Row {i}: key order / names mismatch.\nExpected: {expected}\nGot:      {row_keys}"
            )
    return len(issues) == 0, issues

In [ ]:
# =========================
# EXAMPLE: VALIDATE ORGANIZER JSON SAMPLES
# =========================

for pkg, rows in example_jsons.items():
    ok, issues = validate_output_rows(pkg, rows)
    print(pkg, "->", "VALID" if ok else "INVALID")
    if issues:
        print("\n".join(issues[:2]))

## Suggested participant extensions

Participants can improve this scaffold by adding:

- multilingual OCR
- VLM-based page classification
- stamp/signature/implant sticker detection
- table extraction for invoices / vitals
- robust date normalization to `DD-MM-YYYY`
- exact STG rule encoding per package
- evidence spans and bounding boxes
- calibrated confidence scoring
- duplicate / redundant / extra document tagging
- page grouping into document-level clusters

In [ ]:
# =========================
# DATE NORMALIZATION UTIL
# =========================

from datetime import datetime

def normalize_date(date_str: Optional[str]) -> Optional[str]:
    if not date_str:
        return None

    candidates = [
        "%d/%m/%y", "%d/%m/%Y",
        "%d-%m-%y", "%d-%m-%Y",
        "%d-%b-%y", "%d-%b-%Y",
        "%d %b %Y", "%d %B %Y",
        "%m/%d/%y", "%m/%d/%Y",  # keep only if your data needs it
    ]

    for fmt in candidates:
        try:
            dt = datetime.strptime(date_str, fmt)
            return dt.strftime("%d-%m-%Y")
        except Exception:
            continue
    return date_str

In [ ]:
# =========================
# OPTIONAL: POST-PROCESS DATES INTO REQUIRED FORMAT
# =========================

def normalize_dates_in_rows(package_code: str, rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    date_keys = []
    if package_code == "MG006A":
        date_keys = ["pre_date", "post_date"]
    elif package_code == "SB039A":
        date_keys = ["doa", "dod"]

    normalized = []
    for row in rows:
        r = dict(row)
        for dk in date_keys:
            if dk in r:
                r[dk] = normalize_date(r.get(dk))
        normalized.append(r)
    return normalized

In [ ]:
# =========================
# DATE NORMALIZATION UTIL
# =========================

from datetime import datetime

def normalize_date(date_str: Optional[str]) -> Optional[str]:
    '''
    Normalize a raw date string into the required organizer output format.

    Expected behavior when implemented:
    - accept multiple common date formats extracted from OCR/report text
    - convert valid dates into a standardized output representation
    - return None when no valid date can be parsed reliably
    '''
    pass


In [ ]:
# =========================
# POST-PROCESS DATES INTO REQUIRED FORMAT
# =========================

def normalize_dates_in_rows(package_code: str, rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    '''
    Apply date normalization only to the package-specific date fields.

    Expected behavior when implemented:
    - identify which date keys belong to the current package output schema
    - normalize those fields using normalize_date(...)
    - return rows in the same schema/order expected by evaluation
    '''
    pass

In [ ]:
# =========================
# SAMPLE DECISION REPORT RENDERER
# =========================

def render_decision_report(case_result: Dict[str, Any]) -> None:
    '''
    Render a simple reviewer-facing decision card in notebook output.

    Expected behavior when implemented:
    - display case id, package code, final decision, confidence, and reasons
    - keep the presentation concise and easy to review during the hackathon
    '''
    pass

In [ ]:

# =========================
# MAIN RUNNER / FINAL ASSEMBLY
# =========================

PACKAGE_CODE_LOOKUP = {}

cases = discover_cases(DATA_ROOT)

if not cases:
    print(f"No cases found under: {DATA_ROOT.resolve()}")
    print("Add data first, then rerun this section.")
else:
    BATCH_RESULTS = {}
    FINAL_STRICT_OUTPUTS = {}
    FINAL_PAGE_RESULTS = {}
    FINAL_DECISIONS = {}
    FINAL_SUMMARIES = {}
    FINAL_TIMELINES = {}

    for case_id, files in cases.items():
        package_code = PACKAGE_CODE_LOOKUP.get(case_id, None)

        if package_code is None:
            helper_file = DATA_ROOT / case_id / "package_code.txt"
            if helper_file.exists():
                package_code = helper_file.read_text(encoding="utf-8").strip()

        if package_code not in PACKAGE_CODES:
            print(f"Skipping case '{case_id}' because no valid package code was found.")
            continue

        page_results = []
        strict_rows = []

        for file_path in files:
            extracted_pages = extract_pages(file_path) or []

            for page in extracted_pages:
                page_number = page.get("page_number", 1)
                page_image = page.get("image", None)
                file_name = page.get("file_name", file_path.name)

                page_result = PageResult(
                    case_id=case_id,
                    file_name=file_name,
                    page_number=page_number,
                )

                # OCR
                ocr_output = run_ocr(page_image)
                if isinstance(ocr_output, dict):
                    page_result.extracted_text = ocr_output.get("text", "") or ""
                    page_result.ocr_lines = ocr_output.get("ocr_lines", []) or []

                # Quality
                quality_output = estimate_page_quality(page_image, page_result.extracted_text)
                if isinstance(quality_output, dict):
                    page_result.quality = quality_output

                # Visual cues
                visual_output = detect_visual_elements(page_image, page_result.extracted_text)
                if isinstance(visual_output, dict):
                    page_result.visual_tags = visual_output

                # Document classification
                doc_output = classify_document_type(package_code, page_result.extracted_text, page_result.visual_tags)
                if isinstance(doc_output, dict):
                    page_result.doc_type = doc_output.get("doc_type", "unknown") or "unknown"
                    page_result.doc_type_confidence = float(doc_output.get("confidence", 0.0) or 0.0)

                # Exact row mapping
                output_row = populate_row_for_package(package_code, page_result)
                if isinstance(output_row, dict):
                    page_result.output_row = output_row
                    strict_rows.append(output_row)
                else:
                    # Defensive fallback so the notebook always emits schema-aligned rows
                    schema = PACKAGE_SCHEMAS[package_code]
                    fallback_row = {}
                    for key in schema:
                        if key == "case_id":
                            fallback_row[key] = case_id
                        elif key in {"link", "S3_link", "S3_link/DocumentName", "s3_link"}:
                            fallback_row[key] = file_name
                        elif key == "procedure_code":
                            fallback_row[key] = package_code
                        elif key == "page_number":
                            fallback_row[key] = page_number
                        elif key in {"pre_date", "post_date", "doa", "dod"}:
                            fallback_row[key] = None
                        else:
                            fallback_row[key] = 0
                    page_result.output_row = fallback_row
                    strict_rows.append(fallback_row)

                page_results.append(page_result)

        timeline = build_episode_timeline(package_code, page_results)
        if timeline is None:
            timeline = []

        decision = run_rules_engine(case_id, package_code, strict_rows, timeline)
        if decision is None:
            decision = ClaimDecision(
                case_id=case_id,
                package_code=package_code,
                decision="CONDITIONAL",
                confidence=0.0,
                reasons=["Rules engine not yet implemented."],
            )

        summary_df = build_human_readable_summary(package_code, page_results, decision)
        if summary_df is None:
            summary_df = pd.DataFrame([
                {
                    "case_id": pr.case_id,
                    "file_name": pr.file_name,
                    "page_number": pr.page_number,
                    "doc_type": pr.doc_type,
                    "notes": "Summary builder not yet implemented."
                }
                for pr in page_results
            ])

        timeline_df = build_timeline_df(timeline)
        if timeline_df is None:
            timeline_df = pd.DataFrame([
                {
                    "sequence": t.sequence,
                    "event_type": t.event_type,
                    "date": t.date,
                    "source_document": t.source_document,
                    "temporal_validity": t.temporal_validity,
                }
                for t in timeline
            ])

        case_result = {
            "case_id": case_id,
            "package_code": package_code,
            "page_results": page_results,
            "strict_rows": strict_rows,
            "timeline": timeline,
            "decision": decision,
            "summary_df": summary_df,
            "timeline_df": timeline_df,
        }

        BATCH_RESULTS[case_id] = case_result
        FINAL_STRICT_OUTPUTS[case_id] = strict_rows
        FINAL_PAGE_RESULTS[case_id] = [asdict(pr) for pr in page_results]
        FINAL_DECISIONS[case_id] = asdict(decision)
        FINAL_SUMMARIES[case_id] = summary_df
        FINAL_TIMELINES[case_id] = timeline_df

    print(f"Finished processing {len(BATCH_RESULTS)} case(s).")

# =========================
# FINAL RESULTS DISPLAY
# =========================

if "BATCH_RESULTS" in globals() and BATCH_RESULTS:
    print("\nAvailable final objects:")
    print("- BATCH_RESULTS")
    print("- FINAL_STRICT_OUTPUTS")
    print("- FINAL_PAGE_RESULTS")
    print("- FINAL_DECISIONS")
    print("- FINAL_SUMMARIES")
    print("- FINAL_TIMELINES")

    all_rows = []
    for case_id, rows in FINAL_STRICT_OUTPUTS.items():
        for row in rows:
            all_rows.append(row)

    FINAL_STRICT_OUTPUTS_DF = pd.DataFrame(all_rows)
    print("\nCombined strict output preview:")
    display(FINAL_STRICT_OUTPUTS_DF.head())

    decision_rows = []
    for case_id, decision_dict in FINAL_DECISIONS.items():
        decision_rows.append(decision_dict)

    FINAL_DECISIONS_DF = pd.DataFrame(decision_rows)
    print("\nDecision preview:")
    display(FINAL_DECISIONS_DF.head())
else:
    print("No final results available yet.")